In [3]:
import mne
from mne.viz import Brain
import os
import numpy as np
import pickle
import pyarrow

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config import check_paths
from config.config import (HEMI, ROI, FS_SUB, FS_SRC_PATH, COUPLINGS,
                           SUBJECTS, GROUPS, TASKS, TASK_STAGES, TASK_BLOCKS)
from config.paths import FS_FOLDER, SOURCE_DATA_DIR, ERPAC_DIR, ROI_STCS_DIR, ERPAC_FIGS_DIR
from utils.helpers import iterate_dataset
from utils.plotting import plot_group_erpac, plot_group_erpac_timecourse

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorpac import EventRelatedPac

from sklearn.decomposition import PCA

In [8]:
adaptation_nums = []
for group, sub, task, stage, block in iterate_dataset(
        GROUPS,
        SUBJECTS,
        TASKS,
        TASK_STAGES,
        TASK_BLOCKS
    ):
    # Load the ERPAC data for the current subject, group, task stage, and task block
    epochs_folder = os.path.join(SOURCE_DATA_DIR, group, sub, task, stage, block)
    print(f"{group}_{sub}_{task}_{stage}_{block}: {len(os.listdir(epochs_folder))/2}")
    if block == 'adaptation':
        adaptation_nums.append(len(os.listdir(epochs_folder))/2)

Y_s1_pac_sub01_DeCRAT_plan_baseline: 50.0
Y_s1_pac_sub01_DeCRAT_plan_adaptation: 135.0
Y_s1_pac_sub01_DeCRAT_go_baseline: 62.0
Y_s1_pac_sub01_DeCRAT_go_adaptation: 184.0
Y_s1_pac_sub07_DeCRAT_plan_baseline: 47.0
Y_s1_pac_sub07_DeCRAT_plan_adaptation: 135.0
Y_s1_pac_sub07_DeCRAT_go_baseline: 57.0
Y_s1_pac_sub07_DeCRAT_go_adaptation: 184.0
Y_s1_pac_sub10_DeCRAT_plan_baseline: 48.0
Y_s1_pac_sub10_DeCRAT_plan_adaptation: 140.0
Y_s1_pac_sub10_DeCRAT_go_baseline: 57.0
Y_s1_pac_sub10_DeCRAT_go_adaptation: 186.0
Y_s1_pac_sub11_DeCRAT_plan_baseline: 47.0
Y_s1_pac_sub11_DeCRAT_plan_adaptation: 137.0
Y_s1_pac_sub11_DeCRAT_go_baseline: 61.0
Y_s1_pac_sub11_DeCRAT_go_adaptation: 185.0
Y_s1_pac_sub22_DeCRAT_plan_baseline: 47.0
Y_s1_pac_sub22_DeCRAT_plan_adaptation: 136.0
Y_s1_pac_sub22_DeCRAT_go_baseline: 64.0
Y_s1_pac_sub22_DeCRAT_go_adaptation: 183.0
Y_s1_pac_sub24_DeCRAT_plan_baseline: 48.0
Y_s1_pac_sub24_DeCRAT_plan_adaptation: 131.0
Y_s1_pac_sub24_DeCRAT_go_baseline: 63.0
Y_s1_pac_sub24_DeCRAT_g

In [14]:
min(adaptation_nums)/3

37.666666666666664

In [15]:
max(adaptation_nums)/3

63.666666666666664

In [ ]:
# ============================================================
# TEST SETTINGS
# ============================================================

from numpy import block


group = GROUPS[0]
sub = SUBJECTS[group][0]

task = "FTT"
task_stage = "plan"

# Only relevant for DeCRAT
task_block = None

roi_name = "M1"


# ============================================================
# LOAD FSAVERAGE SOURCE SPACE
# ============================================================

src = mne.read_source_spaces(FS_SRC_PATH)
print(src)


# ============================================================
# LOAD ANATOMICAL LABELS
# ============================================================

labels = mne.read_labels_from_annot(
    subject=FS_SUB,
    parc="aparc.a2009s",
    hemi=HEMI,
    subjects_dir=FS_FOLDER,
)

print(f"Loaded {len(labels)} labels")


# ============================================================
# GET LABEL(S) BELONGING TO FUNCTIONAL ROI
# ============================================================

roi_label_names = ROI[roi_name]

# Make sure it is always a list
if isinstance(roi_label_names, str):
    roi_label_names = [roi_label_names]


roi_labels = [
    label
    for label in labels
    if label.name in roi_label_names
]


print(f"\nROI: {roi_name}")
print("Requested labels:", roi_label_names)
print("Found labels:", [label.name for label in roi_labels])


if len(roi_labels) == 0:
    raise ValueError(
        f"No labels found for ROI {roi_name}. "
        f"Check ROI names against aparc.a2009s."
    )


# ============================================================
# COMBINE LABELS IF ROI CONTAINS >1 ANATOMICAL LABEL
# ============================================================

roi_label = roi_labels[0]

for label in roi_labels[1:]:
    roi_label = roi_label + label


print(
    f"Combined ROI contains "
    f"{len(roi_label.vertices)} label vertices"
)


# ============================================================
# LOAD ONE SOURCE-EPOCH FILE
# ============================================================
#
# Adjust ONLY this section to match the way your files
# are currently named/saved.
#
# The important requirement is that `stcs` becomes:
#
# [
#     SourceEstimate(epoch_1),
#     SourceEstimate(epoch_2),
#     ...
# ]

source_path = os.path.join(
        SOURCE_DATA_DIR,
        group,
        sub,
        task,
        task_stage,
        task_block
    )

stc_files = sorted([
        fname
        for fname in os.listdir(source_path)
        if fname.endswith("-lh.stc")
    ])

stcs = [
        mne.read_source_estimate(
            os.path.join(
                source_path,
                fname
            )
        )
        for fname in stc_files
    ]


print(f"Number of epochs: {len(stcs)}")
print(f"One STC shape: {stcs[0].data.shape}")
print(f"Time range: {stcs[0].times[0]:.3f} "
      f"to {stcs[0].times[-1]:.3f} s")

# ============================================================
# PCA-FLIP EXTRACTION
# ============================================================

roi_pca = mne.extract_label_time_course(
    stcs,
    labels=roi_label,
    src=src,
    mode="pca_flip",
    return_generator=False,
)


print("\nRaw PCA-flip output shape:")
print(roi_pca.shape)


# Expected:
#
# (n_epochs, 1, n_times)
#
# Remove singleton ROI dimension
roi_pca = roi_pca[:, 0, :]


print("\nFinal PCA ROI signal:")
print(roi_pca.shape)

# Expected:
# (n_epochs, n_times)


# ============================================================
# OPTIONAL: MEAN-FLIP FOR COMPARISON
# ============================================================

roi_mean = mne.extract_label_time_course(
    stcs,
    labels=roi_label,
    src=src,
    mode="mean_flip",
    return_generator=False,
)

roi_mean = roi_mean[:, 0, :]


# ============================================================
# COMPARE PCA-FLIP AND MEAN-FLIP
# ============================================================

pca_evoked = np.mean(
    roi_pca,
    axis=0,
)

mean_evoked = np.mean(
    roi_mean,
    axis=0,
)


r = np.corrcoef(
    pca_evoked,
    mean_evoked,
)[0, 1]


print("\nComparison")
print("--------------------------------")
print("PCA-flip shape :", roi_pca.shape)
print("Mean-flip shape:", roi_mean.shape)

print(
    f"Correlation between average "
    f"time courses: r = {r:.3f}"
)


# ============================================================
# BASIC QA
# ============================================================

print("\nPCA signal QA")
print("--------------------------------")

print(
    "NaNs:",
    np.isnan(roi_pca).sum()
)

print(
    "Minimum:",
    np.nanmin(roi_pca)
)

print(
    "Maximum:",
    np.nanmax(roi_pca)
)

print(
    "SD:",
    np.nanstd(roi_pca)
)